In [1]:
import pandas as pd
import numpy as np
import os

folder_path = r"D:\Swapnil\Work\Projects\FIFA\Base_Files"

file_path = os.path.join(folder_path, "FIFA2026_schedule_Fixtures.csv")

fixtures = pd.read_csv(file_path)

print(fixtures.shape)
print(fixtures.columns)
print(fixtures.head())

(104, 6)
Index(['date', 'match_number', 'teams', 'group', 'stadium', 'date_dt'], dtype='object')
                     date match_number  \
0  Thursday, 11 June 2026      Match 1   
1  Thursday, 11 June 2026      Match 2   
2    Friday, 12 June 2026      Match 3   
3    Friday, 12 June 2026      Match 4   
4  Saturday, 13 June 2026      Match 5   

                                               teams    group  \
0                              Mexico v South Africa  Group A   
1  Korea Republic v Czechia/Denmark/North Macedon...  Group A   
2  Canada v Bosnia and Herzegovina/Italy/Northern...  Group B   
3                                     USA v Paraguay  Group D   
4                                   Haiti v Scotland  Group C   

               stadium    date_dt  
0  Mexico City Stadium  6/11/2026  
1  Estadio Guadalajara  6/11/2026  
2      Toronto Stadium  6/12/2026  
3  Los Angeles Stadium  6/12/2026  
4       Boston Stadium  6/13/2026  


In [2]:
fixtures.columns = (
    fixtures.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(fixtures.columns)

Index(['date', 'match_number', 'teams', 'group', 'stadium', 'date_dt'], dtype='object')


In [3]:
fixtures["match_date"] = pd.to_datetime(fixtures["date_dt"], errors="coerce").dt.date

In [4]:
def split_teams(x):
    if pd.isna(x):
        return pd.Series([None, None])
    
    x = str(x).strip()
    
    if " v " in x:
        return pd.Series(x.split(" v ", 1))
    elif " vs " in x:
        return pd.Series(x.split(" vs ", 1))
    else:
        return pd.Series([x, None])

fixtures[["team_1", "team_2"]] = fixtures["teams"].apply(split_teams)

print(fixtures[["teams", "team_1", "team_2"]].head(10))

                                               teams          team_1  \
0                              Mexico v South Africa          Mexico   
1  Korea Republic v Czechia/Denmark/North Macedon...  Korea Republic   
2  Canada v Bosnia and Herzegovina/Italy/Northern...          Canada   
3                                     USA v Paraguay             USA   
4                                   Haiti v Scotland           Haiti   
5        Australia v Kosovo/Romania/Slovakia/Türkiye       Australia   
6                                   Brazil v Morocco          Brazil   
7                                Qatar v Switzerland           Qatar   
8                            Côte d'Ivoire v Ecuador   Côte d'Ivoire   
9                                  Germany v Curaçao         Germany   

                                              team_2  
0                                       South Africa  
1  Czechia/Denmark/North Macedonia/Republic of Ir...  
2  Bosnia and Herzegovina/Italy/Northern I

In [5]:
matches_1 = fixtures[[
    "match_date",
    "match_number",
    "team_1",
    "team_2",
    "group",
    "stadium"
]].copy()

matches_1 = matches_1.rename(columns={
    "team_1": "country",
    "team_2": "opponent"
})

matches_2 = fixtures[[
    "match_date",
    "match_number",
    "team_2",
    "team_1",
    "group",
    "stadium"
]].copy()

matches_2 = matches_2.rename(columns={
    "team_2": "country",
    "team_1": "opponent"
})

fifa_matches = pd.concat([matches_1, matches_2], ignore_index=True)

fifa_matches = fifa_matches.dropna(subset=["country"])

fifa_matches["country"] = fifa_matches["country"].astype(str).str.strip()
fifa_matches["opponent"] = fifa_matches["opponent"].astype(str).str.strip()

In [6]:
fifa_matches["stage"] = np.where(
    fifa_matches["group"].astype(str).str.contains("Group", case=False, na=False),
    "Group Stage",
    "Knockout/Other"
)

In [7]:
fifa_matches = fifa_matches[[
    "match_date",
    "match_number",
    "country",
    "opponent",
    "group",
    "stadium",
    "stage"
]]

fifa_matches = fifa_matches.sort_values(
    ["match_date", "match_number", "country"]
).reset_index(drop=True)

print(fifa_matches.shape)
print(fifa_matches.head(20))
print(fifa_matches.isna().sum())

(208, 7)
    match_date match_number  \
0   2026-06-11      Match 1   
1   2026-06-11      Match 1   
2   2026-06-11      Match 2   
3   2026-06-11      Match 2   
4   2026-06-12      Match 3   
5   2026-06-12      Match 3   
6   2026-06-12      Match 4   
7   2026-06-12      Match 4   
8   2026-06-13      Match 5   
9   2026-06-13      Match 5   
10  2026-06-13      Match 6   
11  2026-06-13      Match 6   
12  2026-06-13      Match 7   
13  2026-06-13      Match 7   
14  2026-06-13      Match 8   
15  2026-06-13      Match 8   
16  2026-06-14     Match 10   
17  2026-06-14     Match 10   
18  2026-06-14     Match 11   
19  2026-06-14     Match 11   

                                              country  \
0                                              Mexico   
1                                        South Africa   
2   Czechia/Denmark/North Macedonia/Republic of Ir...   
3                                      Korea Republic   
4   Bosnia and Herzegovina/Italy/Northern Ireland/... 

In [8]:
output_path = os.path.join(folder_path, "Fifa_Matches.xlsx")

fifa_matches.to_excel(output_path, index=False)

print("Saved:", output_path)

Saved: D:\Swapnil\Work\Projects\FIFA\Base_Files\Fifa_Matches.xlsx
